# Developed-Markets External Validation

This notebook evaluates the frozen US research specifications separately in four pre-specified developed markets: the United Kingdom, Australia, Germany, and France. Country selection is based on institutional comparability, complete 2000--2024 coverage, cross-sectional breadth, characteristic availability, and approximately complete next-month-return coverage; no country is selected using model performance.

Each country uses its stable JKP `id` as the security identifier, excludes nano stocks, rank-normalizes characteristics within its own monthly eligible universe, and applies the unchanged 15-year training, 4-year validation, 1-year test schedule. The resulting OOS period is January 2019--December 2024 (72 months).

The chosen model, `HYBRID_LGBM40_DEEPSET40_DYNAMIC_50_50`, is evaluated as the primary specification. Its fixed US-selected construction is preserved by averaging the aligned OOS predictions of `LGBM_40` and `DEEPSET_40_DYNAMIC` with equal weights in every country; foreign-market data are not used to reselect or reweight it. The two component models are reported as benchmarks, while the validation-weighted `HYBRID_LGBM40_DEEPSET40_DYNAMIC` is retained only as a robustness comparator. Every completed country is reported.

## 1. Runtime and project setup

In [1]:
import gc
import importlib
import os
import sys
from pathlib import Path

LOCAL_PROJECT_DIR = Path(r'C:\Users\sandh\OneDrive\Documents\Coding\FDS Project')
LOCAL_DATA_DIR = Path(r'C:\Users\sandh\FDS Research Project\jkp_developed_153_parquet_2000_2024')
DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/Colab Notebooks/FDS Project')
DRIVE_DATA_DIR = DRIVE_PROJECT_DIR / 'jkp_developed_153_parquet_2000_2024'

try:
    from google.colab import drive
except ImportError:
    RUNNING_IN_COLAB = False
    PROJECT_DIR, DATA_DIR = LOCAL_PROJECT_DIR, LOCAL_DATA_DIR
else:
    RUNNING_IN_COLAB = True
    drive.mount('/content/drive', force_remount=False)
    PROJECT_DIR, DATA_DIR = DRIVE_PROJECT_DIR, DRIVE_DATA_DIR

for required in (PROJECT_DIR, PROJECT_DIR / 'src', DATA_DIR):
    if not required.is_dir():
        raise FileNotFoundError(f'Required directory not found: {required}')
os.chdir(PROJECT_DIR)
project_path = str(PROJECT_DIR)
sys.path = [path for path in sys.path if path != project_path]
sys.path.insert(0, project_path)
for module_name in tuple(sys.modules):
    if module_name == 'src' or module_name.startswith('src.'):
        del sys.modules[module_name]
importlib.invalidate_caches()
import src
if PROJECT_DIR.resolve() not in Path(src.__file__).resolve().parents:
    raise RuntimeError(f'Imported src from the wrong location: {src.__file__}')
print('Project directory:', PROJECT_DIR)
print('Country data directory:', DATA_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project directory: /content/drive/MyDrive/Colab Notebooks/FDS Project
Country data directory: /content/drive/MyDrive/Colab Notebooks/FDS Project/jkp_developed_153_parquet_2000_2024


## 2. Frozen country and model configuration

The external validation preserves the model choice made using the US sample. The component models and validation-weighted hybrid must be estimated within each country; the chosen 50/50 model is then constructed mechanically from the component OOS predictions.

In [2]:
from src.config import ExperimentConfig, UniverseConfig
from src.developed_markets import (
    CHOSEN_MODEL_ID, COMPONENT_BENCHMARK_IDS, COUNTRY_NAMES,
    EXTERNAL_MODEL_IDS, REPORT_MODEL_IDS,
)

COUNTRIES = ('GBR', 'AUS', 'DEU', 'FRA')
OUTPUT_DIR = PROJECT_DIR / 'model_runs' / 'developed_markets'
SUMMARY_DIR = OUTPUT_DIR / 'summary'

CONFIGS = {
    country: ExperimentConfig(
        experiment_id=f'external_validation_{country}_v1',
        project_dir=PROJECT_DIR,
        data_path=DATA_DIR / f'jkp_{country}_153_2000_2024.parquet',
        output_dir=OUTPUT_DIR,
        selected_models=EXTERNAL_MODEL_IDS,
        seed=42,
        use_gpu=True,
        universe=UniverseConfig(
            country=country, start_year=2000, end_year=2024,
            security_id_col='id',
        ),
    )
    for country in COUNTRIES
}
for config in CONFIGS.values():
    config.validate()
print({country: str(config.data_path) for country, config in CONFIGS.items()})

{'GBR': '/content/drive/MyDrive/Colab Notebooks/FDS Project/jkp_developed_153_parquet_2000_2024/jkp_GBR_153_2000_2024.parquet', 'AUS': '/content/drive/MyDrive/Colab Notebooks/FDS Project/jkp_developed_153_parquet_2000_2024/jkp_AUS_153_2000_2024.parquet', 'DEU': '/content/drive/MyDrive/Colab Notebooks/FDS Project/jkp_developed_153_parquet_2000_2024/jkp_DEU_153_2000_2024.parquet', 'FRA': '/content/drive/MyDrive/Colab Notebooks/FDS Project/jkp_developed_153_parquet_2000_2024/jkp_FRA_153_2000_2024.parquet'}


## 3. Preflight validation

In [3]:
import pyarrow.parquet as pq
import torch
from src.config import FEATURES_40
from src.models import MODEL_REGISTRY

required_columns = {
    'id', 'eom', 'excntry', 'size_grp', 'me', 'ret_exc_lead1m', *FEATURES_40
}
for country, config in CONFIGS.items():
    if not config.data_path.is_file():
        raise FileNotFoundError(config.data_path)
    columns = set(pq.read_schema(config.data_path).names)
    missing = sorted(required_columns - columns)
    if missing:
        raise ValueError(f'{country} is missing required columns: {missing}')
    if 'permno' in columns:
        print(f'{country}: PERMNO is present but is not used; JKP id is the security key.')
if set(EXTERNAL_MODEL_IDS) - set(MODEL_REGISTRY):
    raise RuntimeError('An external-validation model is not registered.')
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime before model estimation.')
print('GPU:', torch.cuda.get_device_name(0))
print('Country files and model registry: PASS')

GBR: PERMNO is present but is not used; JKP id is the security key.
AUS: PERMNO is present but is not used; JKP id is the security key.
DEU: PERMNO is present but is not used; JKP id is the security key.
FRA: PERMNO is present but is not used; JKP id is the security key.
GPU: NVIDIA A100-SXM4-40GB
Country files and model registry: PASS


## 4. Run or resume required country-level estimations

Each country and estimated specification has an independent experiment directory and signature. Completed annual refits load from disk; incomplete refits resume without altering completed artifacts. The chosen model itself requires no additional fitting because its equal weights are fixed in advance.

In [4]:
from dataclasses import replace
from src.runner import ExperimentRunner

for country, country_config in CONFIGS.items():
    print(f'\n######## {country}: {COUNTRY_NAMES[country]} ########')
    for model_id in EXTERNAL_MODEL_IDS:
        print(f'\n=== {country} | {model_id} ===')
        model_config = replace(country_config, selected_models=(model_id,))
        ExperimentRunner(model_config).run()
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


######## GBR: United Kingdom ########

=== GBR | LGBM_40 ===
Device: cuda

LGBM_40 [ae0f20ba802df85f]
  refit 01/06 | test_year=2019 | status=training | train_n=193,944 | validation_n=45,059 | test_n=11,514


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


  refit 01/06 | test_year=2019 | status=saved | n_predictions=11,514
  refit 02/06 | test_year=2020 | status=training | train_n=187,209 | validation_n=44,984 | test_n=11,578


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


  refit 02/06 | test_year=2020 | status=saved | n_predictions=11,578
  refit 03/06 | test_year=2021 | status=training | train_n=180,407 | validation_n=45,247 | test_n=10,049


...

    epoch 001/100 | train_mse=0.01507748 | val_mse=0.01820459 | best_val_mse=0.01820459 | early_stop=00/10 | 0.8s
    epoch 002/100 | train_mse=0.01454275 | val_mse=0.01855971 | best_val_mse=0.01820459 | early_stop=01/10 | 0.8s
    epoch 003/100 | train_mse=0.01445306 | val_mse=0.01884587 | best_val_mse=0.01820459 | early_stop=02/10 | 0.8s
    epoch 004/100 | train_mse=0.01458691 | val_mse=0.01865813 | best_val_mse=0.01820459 | early_stop=03/10 | 0.8s
    epoch 005/100 | train_mse=0.01432118 | val_mse=0.01814013 | best_val_mse=0.01814013 | early_stop=00/10 | 0.8s
    epoch 006/100 | train_mse=0.01423647 | val_mse=0.01829912 | best_val_mse=0.01814013 | early_stop=01/10 | 0.8s
    epoch 007/100 | train_mse=0.01416153 | val_mse=0.01822958 | best_val_mse=0.01814013 | early_stop=02/10 | 0.8s
    epoch 008/100 | train_mse=0.01418248 | val_mse=0.01813042 | best_val_mse=0.01813042 | early_stop=00/10 | 0.8s
    epoch 009/100 | train_mse=0.01412479 | val_mse=0.01824658 | best_val_mse=0.01813042 

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


    epoch 001/100 | train_mse=0.01473511 | val_mse=0.01906724 | best_val_mse=0.01906724 | early_stop=00/10 | 0.8s
    epoch 002/100 | train_mse=0.01402714 | val_mse=0.01909678 | best_val_mse=0.01906724 | early_stop=01/10 | 0.8s
    epoch 003/100 | train_mse=0.01393030 | val_mse=0.01896582 | best_val_mse=0.01896582 | early_stop=00/10 | 0.8s
    epoch 004/100 | train_mse=0.01412439 | val_mse=0.01905391 | best_val_mse=0.01896582 | early_stop=01/10 | 0.8s
    epoch 005/100 | train_mse=0.01390303 | val_mse=0.01895643 | best_val_mse=0.01895643 | early_stop=00/10 | 0.8s
    epoch 006/100 | train_mse=0.01385729 | val_mse=0.01911368 | best_val_mse=0.01895643 | early_stop=01/10 | 0.8s
    epoch 007/100 | train_mse=0.01382766 | val_mse=0.01898050 | best_val_mse=0.01895643 | early_stop=02/10 | 0.8s
    epoch 008/100 | train_mse=0.01376672 | val_mse=0.01898555 | best_val_mse=0.01895643 | early_stop=03/10 | 0.8s
    epoch 009/100 | train_mse=0.01369126 | val_mse=0.01895632 | best_val_mse=0.01895632 

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


    epoch 001/100 | train_mse=0.01446192 | val_mse=0.02203080 | best_val_mse=0.02203080 | early_stop=00/10 | 0.9s
    epoch 002/100 | train_mse=0.01424978 | val_mse=0.02103599 | best_val_mse=0.02103599 | early_stop=00/10 | 0.9s
    epoch 003/100 | train_mse=0.01408155 | val_mse=0.02103328 | best_val_mse=0.02103328 | early_stop=00/10 | 1.0s
    epoch 004/100 | train_mse=0.01396426 | val_mse=0.02099886 | best_val_mse=0.02099886 | early_stop=00/10 | 1.0s
    epoch 005/100 | train_mse=0.01397096 | val_mse=0.02100580 | best_val_mse=0.02099886 | early_stop=01/10 | 0.8s
    epoch 006/100 | train_mse=0.01384301 | val_mse=0.02100884 | best_val_mse=0.02099886 | early_stop=02/10 | 0.8s
    epoch 007/100 | train_mse=0.01382800 | val_mse=0.02103978 | best_val_mse=0.02099886 | early_stop=03/10 | 0.8s
    epoch 008/100 | train_mse=0.01388643 | val_mse=0.02111095 | best_val_mse=0.02099886 | early_stop=04/10 | 0.8s
    epoch 009/100 | train_mse=0.01387519 | val_mse=0.02099957 | best_val_mse=0.02099886 

## 5. Chosen 50/50 model and country tables

The chosen model is constructed from aligned OOS component predictions using the fixed rule $0.5\widehat{r}^{\mathrm{LGBM}}_{i,t+1}+0.5\widehat{r}^{\mathrm{DeepSet}}_{i,t+1}$. The table reports it first, followed by the component benchmarks and the validation-weighted robustness comparator.

In [5]:
from IPython.display import display
from src.developed_markets import country_comparison

country_tables = {}
for country, config in CONFIGS.items():
    table = country_comparison(config)
    country_tables[country] = table
    print(f'\n{COUNTRY_NAMES[country]}')
    display(table)


United Kingdom


,country,country_name,model_id,model_role,pooled_oos_r2,robust_oos_r2,mean_monthly_rank_ic,rank_ic_newey_west_t_stat,annualized_return,annualized_volatility,sharpe,newey_west_t_stat,max_drawdown,hit_rate,n_months
0,GBR,United Kingdom,HYBRID_LGBM40_DEEPSET40_DYNAMIC_50_50,chosen_model,0.000927,0.001469,0.088204,10.061946,0.327338,0.140001,2.338105,4.057702,-0.275165,0.777778,72
1,GBR,United Kingdom,LGBM_40,component_benchmark,0.001189,0.001634,0.075961,9.389031,0.307052,0.128454,2.390353,4.226540,-0.244730,0.805556,72
2,GBR,United Kingdom,DEEPSET_40_DYNAMIC,component_benchmark,-0.003436,-0.003354,0.081027,9.947264,0.306980,0.122102,2.514128,4.931987,-0.193222,0.805556,72
3,GBR,United Kingdom,HYBRID_LGBM40_DEEPSET40_DYNAMIC,robustness_comparator,-0.001394,-0.001281,0.080525,10.424331,0.309142,0.134043,2.306285,4.228197,-0.228245,0.777778,72



Australia


,country,country_name,model_id,model_role,pooled_oos_r2,robust_oos_r2,mean_monthly_rank_ic,rank_ic_newey_west_t_stat,annualized_return,annualized_volatility,sharpe,newey_west_t_stat,max_drawdown,hit_rate,n_months
0,AUS,Australia,HYBRID_LGBM40_DEEPSET40_DYNAMIC_50_50,chosen_model,0.004742,0.005056,0.101720,8.433739,0.481833,0.186741,2.580219,5.926192,-0.202881,0.847222,72
1,AUS,Australia,LGBM_40,component_benchmark,0.005131,0.005777,0.096857,9.417564,0.468209,0.160294,2.920933,8.458087,-0.136183,0.847222,72
2,AUS,Australia,DEEPSET_40_DYNAMIC,component_benchmark,0.001499,0.001000,0.091752,7.620722,0.410833,0.157076,2.615512,6.437820,-0.148201,0.847222,72
3,AUS,Australia,HYBRID_LGBM40_DEEPSET40_DYNAMIC,robustness_comparator,0.004694,0.005394,0.093719,8.844524,0.411207,0.181382,2.267083,5.519029,-0.210532,0.791667,72



Germany


,country,country_name,model_id,model_role,pooled_oos_r2,robust_oos_r2,mean_monthly_rank_ic,rank_ic_newey_west_t_stat,annualized_return,annualized_volatility,sharpe,newey_west_t_stat,max_drawdown,hit_rate,n_months
0,DEU,Germany,HYBRID_LGBM40_DEEPSET40_DYNAMIC_50_50,chosen_model,0.002679,0.003197,0.083174,10.942711,0.353132,0.139654,2.528619,6.134483,-0.108061,0.819444,72
1,DEU,Germany,LGBM_40,component_benchmark,-0.001261,-0.002089,0.076864,10.412527,0.335310,0.133335,2.514788,6.324801,-0.092525,0.875000,72
2,DEU,Germany,DEEPSET_40_DYNAMIC,component_benchmark,0.001363,0.001739,0.075573,9.487785,0.270320,0.114515,2.360560,5.585071,-0.116724,0.805556,72
3,DEU,Germany,HYBRID_LGBM40_DEEPSET40_DYNAMIC,robustness_comparator,-0.001784,-0.002861,0.075932,9.344044,0.339745,0.131679,2.580110,6.942859,-0.078811,0.791667,72



France


,country,country_name,model_id,model_role,pooled_oos_r2,robust_oos_r2,mean_monthly_rank_ic,rank_ic_newey_west_t_stat,annualized_return,annualized_volatility,sharpe,newey_west_t_stat,max_drawdown,hit_rate,n_months
0,FRA,France,HYBRID_LGBM40_DEEPSET40_DYNAMIC_50_50,chosen_model,0.000024,0.001729,0.095568,8.424457,0.381900,0.142266,2.684409,5.867847,-0.142920,0.819444,72
1,FRA,France,LGBM_40,component_benchmark,-0.000021,-0.000287,0.081710,7.701541,-0.366395,1.696447,-0.215978,-0.551040,-4.512655,0.763889,72
2,FRA,France,DEEPSET_40_DYNAMIC,component_benchmark,-0.000021,-0.001645,0.084743,8.344216,0.324078,0.128606,2.519927,4.638597,-0.149724,0.777778,72
3,FRA,France,HYBRID_LGBM40_DEEPSET40_DYNAMIC,robustness_comparator,0.000014,0.001859,0.087774,7.950985,0.349578,0.149628,2.336316,5.249711,-0.185206,0.833333,72


## 6. Portfolio implementability robustness

For every country, implementability tests are applied to the chosen 50/50 model and its two component benchmarks. The 10% tail portfolio is evaluated under full and ex-microcap universes, equal and value weighting, proportional transaction costs, missing-return stress, and observed-return outlier scenarios.

In [6]:
import pandas as pd
from src.portfolio_robustness import run_portfolio_robustness

robustness_tables = []
for country, config in CONFIGS.items():
    table = run_portfolio_robustness(
        config.run_dir, model_ids=(CHOSEN_MODEL_ID, *COMPONENT_BENCHMARK_IDS)
    )
    table.insert(0, 'country_name', COUNTRY_NAMES[country])
    table.insert(0, 'country', country)
    robustness_tables.append(table)
external_robustness = pd.concat(robustness_tables, ignore_index=True)
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)
external_robustness.to_csv(
    SUMMARY_DIR / 'developed_markets_portfolio_robustness.csv', index=False
)
display(external_robustness)

HYBRID_LGBM40_DEEPSET40_DYNAMIC_50_50: saved portfolio robustness
LGBM_40: saved portfolio robustness
DEEPSET_40_DYNAMIC: saved portfolio robustness
HYBRID_LGBM40_DEEPSET40_DYNAMIC_50_50: saved portfolio robustness
LGBM_40: saved portfolio robustness
DEEPSET_40_DYNAMIC: saved portfolio robustness
HYBRID_LGBM40_DEEPSET40_DYNAMIC_50_50: saved portfolio robustness
LGBM_40: saved portfolio robustness
DEEPSET_40_DYNAMIC: saved portfolio robustness
HYBRID_LGBM40_DEEPSET40_DYNAMIC_50_50: saved portfolio robustness
LGBM_40: saved portfolio robustness
DEEPSET_40_DYNAMIC: saved portfolio robustness


,country,country_name,universe,weighting,mean_monthly_turnover,annualized_turnover,mean_n_eligible,mean_long_coverage,mean_short_coverage,gross_mean_monthly_return,...,outlier_exclude_abs_gt_10_sharpe,outlier_exclude_abs_gt_10_t_stat,outlier_exclude_abs_gt_10_newey_west_t_stat,outlier_exclude_abs_gt_10_max_drawdown,outlier_exclude_abs_gt_10_hit_rate,outlier_exclude_abs_gt_10_n_months,n_market_abs_gt_10,n_selected_abs_gt_10,model_id,model_signature
0,GBR,United Kingdom,EX_MICRO,EQUAL,1.245499,14.945993,382.347222,0.995576,0.992394,0.009736,...,0.723174,1.771408,1.779641,-0.302879,0.625000,72,0,0,DEEPSET_40_DYNAMIC,11e1aa6b2db4e879
1,GBR,United Kingdom,EX_MICRO,VALUE,1.386975,16.643703,382.347222,0.995576,0.992394,-0.002722,...,-0.166685,-0.408293,-0.369453,-0.456476,0.486111,72,0,0,DEEPSET_40_DYNAMIC,11e1aa6b2db4e879
2,GBR,United Kingdom,FULL,EQUAL,1.165521,13.986246,877.458333,0.993817,0.995377,0.025522,...,2.486943,6.091741,4.867910,-0.197361,0.805556,72,0,0,DEEPSET_40_DYNAMIC,11e1aa6b2db4e879
3,GBR,United Kingdom,FULL,VALUE,1.368770,16.425242,877.458333,0.993817,0.995377,-0.001570,...,-0.081725,-0.200184,-0.154929,-0.634052,0.527778,72,0,0,DEEPSET_40_DYNAMIC,11e1aa6b2db4e879
4,GBR,United Kingdom,EX_MICRO,EQUAL,1.140799,13.689590,382.347222,0.996604,0.993903,0.011987,...,0.841015,2.060058,1.949386,-0.303623,0.625000,72,0,0,HYBRID_LGBM40_DEEPSET40_DYNAMIC_50_50,7547db4f6762a503
5,GBR,United Kingdom,EX_MICRO,VALUE,1.273142,15.277705,382.347222,0.996604,0.993903,-0.002093,...,-0.118264,-0.289686,-0.268206,-0.530959,0.486111,72,0,0,HYBRID_LGBM40_DEEPSET40_DYNAMIC_50_50,7547db4f6762a503
6,GBR,United Kingdom,FULL,EQUAL,1.026793,12.321510,877.458333,0.993228,0.995270,0.027278,...,2.338105,5.727164,4.057702,-0.275165,0.777778,72,0,0,HYBRID_LGBM40_DEEPSET40_DYNAMIC_50_50,7547db4f6762a503
7,GBR,United Kingdom,FULL,VALUE,1.224053,14.688639,877.458333,0.993228,0.995270,0.005420,...,0.223523,0.547517,0.454586,-0.665167,0.583333,72,0,0,HYBRID_LGBM40_DEEPSET40_DYNAMIC_50_50,7547db4f6762a503
8,GBR,United Kingdom,EX_MICRO,EQUAL,0.906845,10.882144,382.347222,0.996264,0.993518,0.010087,...,0.764582,1.872836,2.018678,-0.291749,0.638889,72,0,0,LGBM_40,ae0f20ba802df85f
9,GBR,United Kingdom,EX_MICRO,VALUE,1.011646,12.139748,382.347222,0.996264,0.993518,0.002879,...,0.161519,0.395638,0.458049,-0.412452,0.541667,72,0,0,LGBM_40,ae0f20ba802df85f


## 7. Aggregate cross-country comparison

The first table presents all model-country results with an explicit model role. The second reports the chosen 50/50 model's change in Sharpe ratio and annualized return relative to each benchmark or robustness comparator; positive values favour the chosen model.

In [7]:
from src.developed_markets import aggregate_country_comparisons

external_comparison, chosen_model_improvements = aggregate_country_comparisons(
    CONFIGS, SUMMARY_DIR
)
display(external_comparison)
display(chosen_model_improvements)

,country,country_name,model_id,model_role,pooled_oos_r2,robust_oos_r2,mean_monthly_rank_ic,rank_ic_newey_west_t_stat,annualized_return,annualized_volatility,sharpe,newey_west_t_stat,max_drawdown,hit_rate,n_months
0,GBR,United Kingdom,HYBRID_LGBM40_DEEPSET40_DYNAMIC_50_50,chosen_model,0.000927,0.001469,0.088204,10.061946,0.327338,0.140001,2.338105,4.057702,-0.275165,0.777778,72
1,GBR,United Kingdom,LGBM_40,component_benchmark,0.001189,0.001634,0.075961,9.389031,0.307052,0.128454,2.390353,4.226540,-0.244730,0.805556,72
2,GBR,United Kingdom,DEEPSET_40_DYNAMIC,component_benchmark,-0.003436,-0.003354,0.081027,9.947264,0.306980,0.122102,2.514128,4.931987,-0.193222,0.805556,72
3,GBR,United Kingdom,HYBRID_LGBM40_DEEPSET40_DYNAMIC,robustness_comparator,-0.001394,-0.001281,0.080525,10.424331,0.309142,0.134043,2.306285,4.228197,-0.228245,0.777778,72
4,AUS,Australia,HYBRID_LGBM40_DEEPSET40_DYNAMIC_50_50,chosen_model,0.004742,0.005056,0.101720,8.433739,0.481833,0.186741,2.580219,5.926192,-0.202881,0.847222,72
5,AUS,Australia,LGBM_40,component_benchmark,0.005131,0.005777,0.096857,9.417564,0.468209,0.160294,2.920933,8.458087,-0.136183,0.847222,72
6,AUS,Australia,DEEPSET_40_DYNAMIC,component_benchmark,0.001499,0.001000,0.091752,7.620722,0.410833,0.157076,2.615512,6.437820,-0.148201,0.847222,72
7,AUS,Australia,HYBRID_LGBM40_DEEPSET40_DYNAMIC,robustness_comparator,0.004694,0.005394,0.093719,8.844524,0.411207,0.181382,2.267083,5.519029,-0.210532,0.791667,72
8,DEU,Germany,HYBRID_LGBM40_DEEPSET40_DYNAMIC_50_50,chosen_model,0.002679,0.003197,0.083174,10.942711,0.353132,0.139654,2.528619,6.134483,-0.108061,0.819444,72
9,DEU,Germany,LGBM_40,component_benchmark,-0.001261,-0.002089,0.076864,10.412527,0.335310,0.133335,2.514788,6.324801,-0.092525,0.875000,72


,country,country_name,chosen_minus_lgbm_40_sharpe,chosen_minus_lgbm_40_annualized_return,chosen_minus_deepset_40_dynamic_sharpe,chosen_minus_deepset_40_dynamic_annualized_return,chosen_minus_hybrid_lgbm40_deepset40_dynamic_sharpe,chosen_minus_hybrid_lgbm40_deepset40_dynamic_annualized_return
0,AUS,Australia,-0.340714,0.013624,-0.035293,0.071000,0.313136,0.070626
1,DEU,Germany,0.013831,0.017821,0.168058,0.082812,-0.051491,0.013386
2,FRA,France,2.900387,0.748295,0.164482,0.057823,0.348093,0.032323
3,GBR,United Kingdom,-0.052249,0.020286,-0.176023,0.020358,0.031820,0.018196


## 8. Completion checks

In [8]:
expected_months = 72
expected_models_per_country = len(REPORT_MODEL_IDS)
if len(external_comparison) != len(COUNTRIES) * expected_models_per_country:
    raise RuntimeError('The aggregate country/model table is incomplete.')
if external_comparison['n_months'].ne(expected_months).any():
    display(external_comparison.loc[external_comparison['n_months'].ne(expected_months)])
    raise RuntimeError('At least one country/model does not cover all 72 OOS months.')
expected_robustness_models = 1 + len(COMPONENT_BENCHMARK_IDS)
if len(external_robustness) != len(COUNTRIES) * expected_robustness_models * 4:
    raise RuntimeError('The aggregate implementability table is incomplete.')
print('External validation complete for:', list(COUNTRIES))
print('Model-country rows:', len(external_comparison))
print('OOS period: 2019-01 through 2024-12')
print('Summary directory:', SUMMARY_DIR)

External validation complete for: ['GBR', 'AUS', 'DEU', 'FRA']
Model-country rows: 16
OOS period: 2019-01 through 2024-12
Summary directory: /content/drive/MyDrive/Colab Notebooks/FDS Project/model_runs/developed_markets/summary
